# 第3章：基于直方图统计的处理

## 编程实践：手写直方图统计 / 均衡化 / 匹配

| 项目 | 说明 |
|------|------|
| 输入图片 | `lenaface.jpg`（灰度）；`test1.png` + `test2.png`（直方图匹配源/参考） |
| 手写核心 | 直方图统计、CDF、均衡化、直方图匹配 |
| 允许调用 | 仅 `cv_imread` / `cv_imwrite` 图像读写 |
| 对比验证 | 与 OpenCV `cv2.equalizeHist` 结果做数值误差对比 |


## 一、学习目标

1. 掌握如何提取一张图像的**像素直方图**。
2. 掌握**直方图均衡化**的实现方式与效果特点。
3. 掌握**直方图匹配**（源图 + 参考图）的实现方式与效果特点。


## 二、原理与公式

### 2.1 什么是直方图？——给图像拍一张"像素体检 X 光片"

**直方图**就是把"每种亮度的像素各有多少个"做成一张柱状图：
- 横轴：灰度等级（0 纯黑 → 255 纯白）
- 纵轴：该灰度级出现的**频数**（像素个数）

```
hist[k] = 整幅图中像素值恰好等于 k 的个数，   k = 0, 1, 2, ..., 255
```

> 🎓 **零基础直观理解**：想象把全班 26 万个像素（512×512）按"肤色亮度"站队，
> 0 号队伍最黑、255 号队伍最白。直方图就是数一数每支队伍的人数。
> 队伍如果全部挤在左边（0~50）——图像偏暗；
> 队伍全部挤在右边（200~255）——图像过曝；
> 队伍分布很窄（集中在 100~120）——对比度低、灰蒙蒙。

**直方图的性质：**
- ✅ 反映**全局亮度与对比度**分布
- ❌ **丢失空间位置信息**——把像素随机打乱，直方图完全不变！

### 2.2 直方图均衡化：让灰度"排队更均匀"

目标：把原本扎堆（集中在某几个灰级）的像素，通过一个单调映射函数，**摊平**到 0~255 的全范围。
结果：原来对比度过低的图像会变得层次分明、细节清晰。

**三步走公式：**

| 步骤 | 公式 | 说明 |
|------|------|------|
| ① 统计直方图 | $h[k] = \text{count}(I=k)$ | 每个灰级有几个像素 |
| ② 计算 CDF | $CDF[k] = \frac{1}{N} \sum_{i=0}^{k} h[i]$ | 累积分布函数（≤k 的像素占比） |
| ③ 映射 LUT | $\text{LUT}[k] = \mathrm{round}(CDF[k] \times 255)$ | 把累积比例映射回 0~255 |

最后对每个像素查表：$I_{out}[y,x] = \text{LUT}[I_{in}[y,x]]$。

> 🔑 **为什么叫"均衡化"？** 数学上，如果输入是连续随机变量 $X$，
> 那么 $Y = CDF_X(X)$ 一定服从均匀分布 U[0,1]。
> 离散图像近似这个过程，所以 CDF 会被拉成一条近似"从左下到右上的斜线"。

### 2.3 直方图匹配（规定化）："照着别人的色调整容"

**应用场景**：
把一张阴天灰蒙蒙的源图，改成跟一张阳光明媚的参考图**同样的色调风格**——这就是直方图匹配。

**核心思路**（对每个灰级 $i$）：
1. 分别求源图和参考图的 CDF；
2. 在参考 CDF 里找一个 $j$，使得 $CDF_{ref}[j]$ 最接近 $CDF_{src}[i]$；
3. 令映射表 $\text{LUT}[i] = j$。

即：
$$
\text{LUT}[i] = \arg\min_j \; \left| CDF_{ref}[j] - CDF_{src}[i] \right|
$$

然后源图每个像素按 LUT 查表即可。

> 💡 **零基础类比**：源图是 A 班分数分布（大家都扎堆 60 分），
> 参考图是 B 班分数分布（正态分布 60~90）。
> 匹配就是把 A 班的分数"重排"，让分数分布的形状和 B 班一样。



## 三、手写约束清单

- ✅ 允许：`cv_imread` / `cv_imwrite`；Python 循环与算术；`np.zeros` 开辟空间。
- ❌ 禁止：`np.histogram` / `cv2.calcHist` / `cv2.equalizeHist` 用于实现（这些仅可用于对比验证）。
- ✅ 可视化：`matplotlib` 仅用于显示直方图与图像。


In [ ]:
import sys
from pathlib import Path

# 向上查找项目根目录（含 utils.py），并加入 sys.path
ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("未找到项目根目录 utils.py")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results

setup_plot_chinese()
set_random_seed(42)
print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {Path.cwd()}")


In [ ]:
def to_grayscale_manual(image):
    """手写灰度化：Y = 0.299R + 0.587G + 0.114B。"""
    h, w, _ = image.shape
    gray = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            b, g, r = (float(v) for v in image[y, x, :])
            gray[y, x] = int(round(0.299 * r + 0.587 * g + 0.114 * b))
    return gray


def compute_hist_manual(gray):
    """手写直方图统计，返回长度 256 的频数数组。"""
    hist = np.zeros(256, dtype=np.int64)
    for y in range(gray.shape[0]):
        for x in range(gray.shape[1]):
            hist[int(gray[y, x])] += 1
    return hist


def compute_cdf(hist, total):
    """由直方图计算归一化累积分布函数 CDF。"""
    cdf = np.zeros(256, dtype=np.float64)
    acc = 0
    for i in range(256):
        acc += hist[i]
        cdf[i] = acc / total
    return cdf


def histogram_equalization_manual(gray):
    """手写直方图均衡化。"""
    hist = compute_hist_manual(gray)
    total = gray.shape[0] * gray.shape[1]
    cdf = compute_cdf(hist, total)

    lut = np.zeros(256, dtype=np.uint8)
    for i in range(256):
        lut[i] = int(round(cdf[i] * 255.0))

    h, w = gray.shape
    out = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            out[y, x] = lut[int(gray[y, x])]
    return out, hist, lut


def histogram_matching_manual(source, reference):
    """手写直方图匹配：把 source 的直方图逼近 reference 的直方图。"""
    src_hist = compute_hist_manual(source)
    ref_hist = compute_hist_manual(reference)
    src_cdf = compute_cdf(src_hist, source.shape[0] * source.shape[1])
    ref_cdf = compute_cdf(ref_hist, reference.shape[0] * reference.shape[1])

    lut = np.zeros(256, dtype=np.uint8)
    for i in range(256):
        best_j, best_dist = 0, float("inf")
        for j in range(256):
            dist = abs(ref_cdf[j] - src_cdf[i])
            if dist < best_dist:
                best_dist = dist
                best_j = j
        lut[i] = best_j

    h, w = source.shape
    out = np.zeros((h, w), dtype=np.uint8)
    for y in range(h):
        for x in range(w):
            out[y, x] = lut[int(source[y, x])]
    return out, src_hist, ref_hist


In [ ]:
# 可视化直方图与结果：使用教程提供的 show_histogram，自动叠加 CDF 曲线
from utils import show_histogram

show_histogram(
    [gray, eq],
    titles=["原图直方图（叠加 CDF 曲线）", "均衡化后直方图（叠加 CDF 曲线）"],
    colors=["#3498db", "#e67e22"],
    show_cdf=True,
    suptitle="直方图均衡化前后对比：注意 CDF 从曲线变成近似直线",
)

show_images([gray, eq], ["原灰度图", "直方图均衡化结果"], show_info=True, figsize=(9, 4),
            suptitle="肉眼对比：均衡化后对比度增强，细节更清晰")


In [ ]:
# 直方图匹配：源/参考/匹配后 三者直方图并排对比，带 CDF 曲线
from utils import show_histogram

show_histogram(
    [src, ref, matched],
    titles=["源图直方图", "参考图直方图", "匹配后直方图"],
    colors=["#3498db", "#e74c3c", "#27ae60"],
    show_cdf=True,
    suptitle="直方图匹配：观察匹配后的曲线（右）形状是否逼近参考图（中）",
)

show_images([src, ref, matched], ["源图", "参考图", "直方图匹配结果"], show_info=True,
            figsize=(13, 4), suptitle="肉眼对比：匹配后源图的色调风格向参考图靠拢")


## 四、结果与参数分析

- 均衡化把 CDF 线性化，整幅图对比度增强；直方图会从聚集变为**更分散**。
- 匹配让源图灰度分布逼近参考图，可理解为"按参考图的色调风格重映射源图"。
- 直方图方法只改变灰度映射（**像素值重映射**），不改变像素位置，因此不引入几何失真。

**易错点**
1. CDF 必须用**总像素数**归一化，否则 LUT 溢出。
2. 匹配时的最近邻搜索要处理 CDF 相等的情况，优先取较小的 `j`。
3. 彩色图应先分离通道或转灰度再处理，避免把三通道混在一起统计。


## 五、科研规范小结

1. **统计量与算法分离**：`compute_hist_manual` / `compute_cdf` 各自单一职责。
2. **可复现**：直方图方法本身无随机性，但统一使用 `set_random_seed` 便于与其它章节保持一致。
3. **对比验证**：与 `cv2.equalizeHist` 做数值对比，误差应为 0 或仅因取整产生 1 个灰度级。


## 六、练习：彩色图三通道分别均衡化

**要求**：对 `lena.jpeg` 的 B、G、R 三通道分别做直方图均衡化，再合并显示；观察与"转灰度后均衡化"的差异。


In [ ]:
# ==================== 练习解决方案 ====================
def equalize_color_manual(image):
    """对 BGR 三通道分别做直方图均衡化。"""
    h, w, _ = image.shape
    out = np.zeros_like(image)
    for ch in range(3):
        out[:, :, ch] = histogram_equalization_manual(image[:, :, ch])[0]
    return out

img = cv_imread("lena.jpeg", cv2.IMREAD_COLOR)
out_color = equalize_color_manual(img)
show_images([img, out_color], ["原图", "三通道分别均衡化"], figsize=(9, 4))
